In [1]:
# Run this in a new cell to disable MLflow logging
!yolo settings mlflow=False

✅ Updated 'mlflow=False'
JSONDict("/home/sagemaker-user/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/home/sagemaker-user/42174_AI_Studio_Hajimi/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "d988cf958e79cc84710a2ebbc5929cd65b4a57cd969ccf903f9b8820ee1ad242",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": false,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [2]:
import os
import yaml
import shutil
from ultralytics import YOLO

#  Automated Dataset Path Discovery 
# We use recursive searching to find the 'train' and 'val' folders 
# regardless of how the .zip file was nested.
search_root = os.path.abspath('./dataset/unzipped_full_dataset')
real_data_root = ""

print("Scanning directory structure for training entry points...")
for root, dirs, files in os.walk(search_root):
    # Identifying the root directory that contains both 'train' and 'val' folders
    if 'train' in dirs and 'val' in dirs:
        real_data_root = root
        break

if not real_data_root:
    raise FileNotFoundError(" Critical Error: Could not locate 'train' or 'val' directories. Please check zip content.")

print(f"Target dataset root identified at: {real_data_root}")

# Intelligent Sub-directory Detection
# Determining if the structure follows 'train/images' or just 'train/'
has_images_subdir = os.path.exists(os.path.join(real_data_root, 'train/images'))

train_path = 'train/images' if has_images_subdir else 'train'
val_path = 'val/images' if has_images_subdir else 'val'

# Dynamic Data Configuration Generation ---
# Creating the data.yaml file with absolute paths for maximum stability
data_config = {
    'path': real_data_root,
    'train': train_path,
    'val': val_path,
    'names': {0: 'Business Card'}
}

yaml_path = os.path.abspath('data_final.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Training configuration (YAML) generated at: {yaml_path}")

# Launching High-Performance Training ---
print("\n Paths aligned. Initializing Full Power Training on Tesla T4...")

# Loading the pre-trained Small model as the backbone
model = YOLO('yolov8s.pt')

# Starting the fine-tuning process
# epochs=150: Sufficient iterations for high-density datasets
# imgsz=1024: Maintaining high-resolution for small text/edge detection
# batch=8: Optimized for 15GB VRAM (Reduce to 4 if OOM occurs)
model.train(
    data=yaml_path,
    epochs=150,
    imgsz=1024,
    batch=8,           
    workers=4,          # Multi-threaded data loading
    name='Hajimi_Final_Battle',
    project='Training_Log',
    exist_ok=True       # Overwrite existing runs to save storage space
)

print(f"\n Training Complete! Best weights saved at: Training_Log/Hajimi_Final_Battle/weights/best.pt")

Scanning directory structure for training entry points...
Target dataset root identified at: /home/sagemaker-user/42174_AI_Studio_Hajimi/dataset/unzipped_full_dataset/yolo_split
Training configuration (YAML) generated at: /home/sagemaker-user/42174_AI_Studio_Hajimi/data_final.yaml

 Paths aligned. Initializing Full Power Training on Tesla T4...
Ultralytics 8.4.31 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14918MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sagemaker-user/42174_AI_Studio_Hajimi/data_final.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hal

E0000 00:00:1774789431.022640   32073 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774789431.028169   32073 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


ClearML results page: https://app.clear.ml/projects/444fed92a7e4445d8c6794657d7168da/experiments/184e999aed8243788bbd247949ed3b65/output/log
WARNING ⚠️ ClearML Initialized a new task. If you want to run remotely, please add clearml-init and connect your arguments before initializing YOLO.
Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True

In [3]:
import os
from ultralytics import YOLO

# Load the Fine-Tuned Model 
# Path sourced from the final training logs
# This 'best.pt' contains the optimized weights after 150 training epochs.
best_model_path = 'runs/detect/Training_Log/Hajimi_Final_Battle/weights/best.pt'
model = YOLO(best_model_path)

# Execute Final Inference (Validation) 
# Running the model on the 8 processed images using the high-definition (1024px) setting.
print(f" Validating with Trained Model: {best_model_path}")

results = model.predict(
    source='./processed_dataset',
    imgsz=1024,         
    conf=0.3,           
    save=True,          
    project='test-Result', 
    name='CNN-Final_Trained_Result', 
    exist_ok=True      
)

print(f"Validation Complete! Results saved in: 'test-Result/CNN-Final_Trained_Result'")

 Validating with Trained Model: runs/detect/Training_Log/Hajimi_Final_Battle/weights/best.pt

image 1/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/113.jpg: 1024x1024 1 Business Card, 21.1ms
image 2/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/119.jpg: 1024x1024 2 Business Cards, 21.1ms
image 3/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/175.jpg: 1024x1024 (no detections), 21.1ms
image 4/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/179.jpg: 1024x1024 1 Business Card, 21.1ms
image 5/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/228.jpg: 1024x1024 2 Business Cards, 21.1ms
image 6/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/279.jpg: 1024x1024 2 Business Cards, 21.1ms
image 7/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/286.jpg: 1024x1024 (no detections), 21.0ms
image 8/8 /home/sagemaker-user/42174_AI_Studio_Hajimi/processed_dataset/289.jpg: 1024x1024 1 Busin